# AGENT

In [1]:
import os
import io
import base64
import PIL
from PIL import Image
from openai import OpenAI
from dotenv import load_dotenv
import requests
from urllib.parse import urlparse
import argparse
import sys
import yaml
import dspy
from tqdm import tqdm
import torch
from sentence_transformers import SentenceTransformer

load_dotenv()

# Initialize the OpenAI Client
client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("BASE_URL")
)

# lm = dspy.LM(f"openai/{os.getenv("TEXT_MODEL")}", api_key=os.getenv("DASHSCOPE_API_KEY"), base_url=os.getenv("BASE_URL"))
# dspy.configure(lm=lm)
lm = dspy.LM(f"openai/{os.getenv("VISION_MODEL")}", api_key=os.getenv("DASHSCOPE_API_KEY"), base_url=os.getenv("BASE_URL"))
dspy.configure(lm=lm)

Skipping import of cpp extensions due to incompatible torch version 2.8.0 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info
W0113 13:56:55.457000 56448 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
from src.utils import create_layout_dataframe
from src.image2embeddings import create_embedding_index
from src.embeddings2clusters import cluster_embeddings
from src.utils import get_segmentation_crops
from src.utils import pdf_to_base64_pngs, base64_to_pillow
import pandas as pd

df = pd.read_parquet("chunk_embeddings.parquet")
# File Paths
GEOJSON_FILE = "chunked_data/冷冻机房0327_t3.geojson"
TRANSFORM_FILE = "images/冷冻机房0327_t3_transform.json"
IMAGE_FILE   = "images/冷冻机房0327_t3_OVERLAY.png"
INDEX_FILE = 'chunk_embeddings.parquet'

df = create_layout_dataframe(geojson_path=GEOJSON_FILE, transform_path=TRANSFORM_FILE)
create_embedding_index(df, IMAGE_FILE, INDEX_FILE)
clusters = cluster_embeddings(INDEX_FILE, visualize=False, method='agglomerative')

[INFO] Processing 401 features...
--- 1. Identify Unique Chunks ---
--- 2. Processing 21 chunks ---


Encoding: 100%|█████████████████████████████████| 21/21 [00:22<00:00,  1.08s/it]
/opt/anaconda3/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/opt/anaconda3/lib/python3.12/site-packages/umap/umap_.py:2462: UserWarning: n_neighbors is larger than the dataset size; truncating to X.shape[0] - 1
  warn(


--- Success! Saved 21 vectors to chunk_embeddings.parquet ---
Reducing 21 embeddings with UMAP...
Clustering using AGGLOMERATIVE...
Found 4 clusters.


In [3]:
from src.retrieve_pages import retrieve_page_indices

b64s = pdf_to_base64_pngs("BuildingCodes/2021_International_Building_Code.pdf")
df_embeds = pd.read_parquet("building_codes_embeddings.parquet")

# # Retrieve top matching page indices
# indices = retrieve_page_indices(
#     query="屋顶光伏安装的要求",
#     top_k=7,
#     top_p=0.95
# )

# print("Top matching pages (index, pdf, page number, score):")
# for idx in indices:
#     row = df_embeds.iloc[idx]
#     print(f"Index {idx}: {row['pdf_source']} - Page {row['page_number']}")

Processing pages: 100%|███████████████████████| 833/833 [00:16<00:00, 51.48it/s]


In [4]:
import dspy
from typing import List, Optional
from PIL import Image

def search_building_codes(query: str, top_k: int = 5, top_p: float = 0.95) -> List[dspy.Image]:
    """
    Retrieve the most relevant building code pages as images based on the query.

    This function uses the retrieval system (retrieve_page_indices) to find the top matching page indices
    for the given query, then converts the corresponding base64-encoded page images to PIL Image objects.
    These images can be analyzed visually (e.g., by a multimodal LLM) to extract text, titles, diagrams,
    or specific code sections.

    Parameters
    ----------
    query : str
        The search query describing the building code topic or requirement (e.g., "屋顶光伏安装的要求").
    top_k : int, optional
        Number of top-relevant pages to retrieve (default: 5).
    top_p : float, optional
        Top-p (nucleus) threshold used by the retriever, if applicable (default: 0.95).

    Returns
    -------
    List[Image.Image]
        List of PIL Images of the retrieved building code pages.
    """
    indices = retrieve_page_indices(query=query, top_k=top_k, top_p=top_p)
    images = [dspy.Image.from_PIL(base64_to_pillow(b64s[i])) for i in indices]
    return images


class CADAnalyzer(dspy.Signature):
    """
    Analyze a CAD drawing image to produce a clear, structured textual description
    of its content, suitable for engineering review or defect detection.

    If user_query is provided, focus the description on aspects relevant to the query.
    The narrative language of the response must match the language of user_query, while
    keeping all drawing labels exactly as shown in the image.
    """

    user_query: Optional[str] = dspy.InputField(
        default=None,
        desc="Optional user query to guide/focus the analysis. If provided, the response language must match the query, while keeping all drawing labels exactly as shown."
    )

    image: dspy.Image = dspy.InputField(
        desc="The CAD drawing image (full view or high-resolution section)."
    )

    response: str = dspy.OutputField(
        desc=(
            "A concise, structured description of the drawing. "
            "Cover: overall type/purpose, scale/units/title block, main systems, "
            "key areas/rooms/equipment, major elements (walls, beams, piping, symbols, etc.), "
            "annotations/dimensions/legends, language(s) used, and any revisions/stamps. "
            "Use exact labels from the drawing. Do not translate labels, do not paraphrase labels, and do not invent details."
        )
    )

USER_QUERY = None

analyzer = dspy.ChainOfThought(CADAnalyzer)
cad_agent = dspy.ReAct(CADAnalyzer, tools=[search_building_codes], max_iters=3)
output = cad_agent(user_query=USER_QUERY, image=dspy.Image.from_file(IMAGE_FILE))
# output = analyzer(image=dspy.Image.from_file(IMAGE_FILE))

# class CADAgent(dspy.Module):
#     def __init__(self):
#         analyzer = dspy.ChainOfThought(CADAnalyzer)
#         self.agent = dspy.ReAct(analyzer, tools=[search_building_codes], max_iters=3)

#     def forward(self):
#         return self.agent()
        

# # Updated instructions to be general (not tied to "foundation") and to reflect that pages come as images
# instructions = (
#     "You are analyzing building code pages provided as images. "
#     "Find and list all section titles (or subsection headings) that are relevant to verifying or refuting the claim. "
#     "If needed, call the tool multiple times with refined queries to retrieve additional pages. "
#     "Only include titles that directly relate to the claim."
# )

# signature = dspy.Signature(
#     "claim -> titles: list[str]",
#     instructions=instructions
# )

# react = dspy.ReAct(signature, tools=[search_building_codes], max_iters=3)

# # # Example usage (testing the retrieval tool directly)
# # imgs = search_building_codes(query="屋顶光伏安装的要求")

In [8]:
# react(claim="屋顶光伏安装的要求").titles
# Image.open(io.BytesIO(base64.b64decode()))
# output['trajectory'], output['reasoning'], output['response']
output['trajectory']

{'thought_0': 'The image shows two architectural floor plans for a refrigeration (cold/hot) machine room, with detailed layouts including dimensions, room functions, and equipment placements. The left side appears to be the original design, while the right side shows modifications or an updated version, indicated by different colors and annotations. To provide a comprehensive analysis, I need to verify relevant building codes related to such facilities, especially regarding safety, ventilation, and equipment placement.',
 'tool_name_0': 'search_building_codes',
 'tool_args_0': {'query': '制冷机房设计规范和安全要求', 'top_k': 5},
 'observation_0': [Image(url=data:image/png;base64,<IMAGE_BASE_64_ENCODED(487920)>),
  Image(url=data:image/png;base64,<IMAGE_BASE_64_ENCODED(486016)>),
  Image(url=data:image/png;base64,<IMAGE_BASE_64_ENCODED(507868)>),
  Image(url=data:image/png;base64,<IMAGE_BASE_64_ENCODED(431660)>),
  Image(url=data:image/png;base64,<IMAGE_BASE_64_ENCODED(464292)>)],
 'thought_1': 'The

In [6]:
def coverage_ratio(roi_img, layout_img):
    """
    Computes area coverage of ROI relative to layout image.
    Assumes both are PIL Images.
    """
    roi_area = roi_img.width * roi_img.height
    layout_area = layout_img.width * layout_img.height

    return roi_area / layout_area

# CAD Analysis

In [7]:
from typing import Optional

class CADAnalyzer(dspy.Signature):
    """
    Perform a comprehensive, structured analysis of a CAD drawing to produce a detailed,
    technically accurate description suitable for engineering review, defect detection,
    or documentation purposes.

    The output should cover:
    - Overall purpose and type of the drawing (e.g., architectural floor plan,
      mechanical layout, structural foundation, piping schematic, etc.)
    - Scale, units, and title block information
    - Main systems or disciplines shown (structural, mechanical, electrical, plumbing, etc.)
    - Key areas, rooms, zones, or equipment groups with their labels and functions
    - Major elements: walls, columns, beams, equipment, piping/duct runs, symbols
    - Notable annotations, dimensions, legends, and references to other drawings
    - Language(s) used in text and labels
    - Any visible revisions, stamps, or approval notes

    If user_query is provided, focus the description on aspects relevant to the query.
    The narrative language of the response must match the language of user_query, while
    keeping all drawing labels exactly as shown in the image.

    Write in clear, professional engineering language. Be specific and exhaustive
    while remaining concise. Use actual labels and terms from the drawing.
    """

    user_query: Optional[str] = dspy.InputField(
        default=None,
        desc=(
            "Optional user query to guide/focus the analysis. If provided, the response language "
            "must match the query, while keeping all drawing labels exactly as shown."
        )
    )

    image: dspy.Image = dspy.InputField(
        desc=(
            "The full CAD drawing image (or a high-resolution crop of a layout/section). "
            "May contain technical symbols, dimensions, annotations in any language, "
            "hatched areas, layered elements, and legends."
        )
    )

    response: str = dspy.OutputField(
        desc=(
            "A detailed, structured textual description of the entire visible content "
            "in the CAD drawing. Organize logically (e.g., start with overview → scale → "
            "major zones → systems → notable details). Include precise labels, dimensions, "
            "and technical terms exactly as they appear. Do not translate labels, do not paraphrase "
            "labels, and do not invent details. "
            "Aim for completeness suitable as input for downstream defect analysis."
        )
    )

analyzer = dspy.ChainOfThought(CADAnalyzer)

USER_QUERY = None

output = analyzer(user_query=USER_QUERY, image=dspy.Image.from_file(IMAGE_FILE))

In [8]:
output.response

'The provided image contains two CAD drawings of a refrigeration (and heating) machine room, presented side-by-side for comparative analysis. Both plans are titled "冷冻(制热)机房平面图" (Refrigeration/Heating Machine Room Plan) and drawn at a scale of 1:50.\n\n**Left Drawing (Original Layout):**\n- **Overall Layout:** Rectangular building footprint measuring approximately 16,000 mm × 8,000 mm.\n- **Main Zones:**\n  - **值班室 (Control Room)** – Located in the upper right corner, shaded purple.\n  - **前室 (Anteroom)** – Adjacent to the control room, also shaded purple.\n  - **走廊 (Corridor)** – Runs along the top edge and central axis, connecting major areas.\n  - **冷冻(制热)机房 (Refrigeration/Heating Machine Room)** – Main mechanical area occupying most of the space, shaded green.\n- **Equipment & Systems:**\n  - Multiple rectangular blocks labeled “S/A” (likely Air Handling Units or Chillers), each with associated dimensions.\n  - Refrigerant types indicated: R410A, R22, suggesting multiple cooling ci

# Layout analysis

In [9]:
from typing import Optional

class LayoutAnalyzer(dspy.Signature):
    """
    Perform a focused, detailed analysis of a specific layout extracted from a larger CAD drawing.
    
    You are provided with:
    - A full description of the entire CAD drawing for global context
    - The complete original drawing image for reference
    - A cropped or segmented image containing only the target layout
    
    Your task is to describe this specific layout in depth, while understanding its role within the broader drawing.
    
    The output should include:
    - Clear identification of the layout type (e.g., Floor Plan, Foundation Plan, Piping Layout, Equipment Schedule, Section View)
    - Scale, orientation, and any title or label identifying this layout
    - Primary discipline/focus (architectural, structural, mechanical, electrical, plumbing, etc.)
    - Key zones, rooms, areas, or equipment groups present, with exact labels as shown
    - Major visible systems: foundations, walls, columns, piping runs, ducts, drainage, electrical routes, etc.
    - Notable symbols, hatches, line types, and their meanings (refer to legend if visible)
    - Critical dimensions, elevations, or annotations unique to this layout
    - Relationships to adjacent areas or references to other drawings/layouts
    - Any revision marks, notes, or special instructions specific to this section
    - Language(s) used in text and labels

    If user_query is provided, focus the description on aspects relevant to the query.
    The narrative language of the response must match the language of user_query, while
    keeping all layout labels exactly as shown in the image.

    Write in clear, technical language. Be exhaustive about what is visible in the layout image, 
    but concise and structured. This description will be used for downstream defect detection in segmented regions.
    """

    user_query: Optional[str] = dspy.InputField(
        default=None,
        desc=(
            "Optional user query to guide/focus the layout analysis. If provided, the response language "
            "must match the query, while keeping all labels exactly as shown."
        )
    )

    description: str = dspy.InputField(
        desc=(
            "A comprehensive textual description of the entire original CAD drawing. "
            "Use this to understand the overall project context, purpose, systems, and terminology "
            "before analyzing the specific layout."
        )
    )

    full_image: dspy.Image = dspy.InputField(
        desc=(
            "The complete original CAD drawing image. Provided for global spatial context, "
            "orientation, and to cross-reference elements visible at the edges of the layout crop."
        )
    )

    layout_image: dspy.Image = dspy.InputField(
        desc=(
            "The extracted or cropped image containing only the target layout. "
            "This is the primary focus — analyze all visible content in detail: labels, dimensions, "
            "symbols, lines, hatches, annotations, and geometry."
        )
    )

    response: str = dspy.OutputField(
        desc=(
            "A detailed, structured description of the specific layout shown in layout_image. "
            "Organize logically (e.g., start with layout type and scale → major zones → systems → key details). "
            "Use exact labels and terms from the drawing. Do not translate labels, do not paraphrase "
            "labels, and do not invent details. "
            "Highlight any unique features, references, or potential areas of concern that may require "
            "closer inspection in downstream chunk analysis."
        )
    )

In [10]:
# layout_output.response

In [11]:
from collections import defaultdict
from typing import Dict, List, Any

LAYOUT_OUTPUTS: Dict[int, List[Any]] = defaultdict(list)

USER_QUERY = None

for l in df['layout_id'].unique():
    raw_img, overlay_img = get_segmentation_crops(
        df,
        IMAGE_FILE,
        target_layouts=[l],
    )

    layout_analyzer = dspy.ChainOfThought(LayoutAnalyzer)
    layout_output = layout_analyzer(
        user_query=USER_QUERY,
        description=output.response,
        full_image=dspy.Image.from_file(IMAGE_FILE),
        layout_image=dspy.Image.from_PIL(raw_img),
    )

    LAYOUT_OUTPUTS[l].append(layout_output)

In [12]:
# print(LAYOUT_OUTPUTS[1][0].response)

# Design defect analysis

In [13]:
from enum import Enum
from typing import List, Optional
from pydantic import BaseModel, Field, ConfigDict
import dspy


class Severity(str, Enum):
    LOW = "Low"
    MEDIUM = "Medium"
    HIGH = "High"


class Defect(BaseModel):
    id: int = Field(..., description="Unique identifier for the defect (sequential integer across the entire report)")
    type: str = Field(..., description="Category of the defect, e.g., 'Unit Mismatch', 'Improper Layer Usage'")
    description: str = Field(..., description="Detailed explanation of the issue")
    location: Optional[str] = Field(
        None,
        description="Where in the drawing the issue occurs (e.g., 'Layer: A-WALL', 'View: Section A-A', 'Coordinates: X=10, Y=20')"
    )
    severity: Severity = Field(..., description="Impact level: Low, Medium, or High")
    how_to_fix: str = Field(..., description="Recommended steps to resolve the defect")

    # Provenance fields for traceability
    layout_id: str = Field(..., description="Identifier of the layout this defect belongs to")
    chunk_id: Optional[str] = Field(None, description="Identifier of the specific chunk/ROI where the defect was detected")
    cluster_id: Optional[str] = Field(None, description="Optional cluster/group ID if defects span multiple chunks")

    model_config = ConfigDict(
        extra="forbid"  # Prevent extra fields
    )


class CADDefectReport(BaseModel):
    drawing_summary: str = Field(..., description="Brief overview of the analyzed CAD drawing")
    defects: List[Defect] = Field(
        default_factory=list,
        description="List of detected design defects. Empty list if no defects found."
    )
    overall_recommendations: Optional[str] = Field(
        None,
        description="General advice for improving the drawing or preventing future issues"
    )
    sources: Optional[List[str]] = Field(
        default_factory=list,
        description="List of references or sources used for detection logic"
    )

    model_config = ConfigDict(
        json_encoders={Severity: lambda v: v.value},  # Custom encoder for Severity enum
        use_enum_values=True,                         # Serialize enums as their values
        extra="allow"  # or "forbid" if you want strictness here too
    )


class DesignDefectAnalysis(dspy.Signature):
    """
    Detect all design defects in a specific region (chunk) of a CAD drawing layout.
    Return a list of defects found in this chunk. If none, return empty list.
    """
    description: str = dspy.InputField(desc="Overall description of the full CAD drawing")
    layout_description: str = dspy.InputField(desc="Description of the current layout being analyzed")
    layout_image: dspy.Image = dspy.InputField(desc="Full layout image for context")
    cluster: dspy.Image = dspy.InputField(desc="Expanded cluster/region containing multiple chunks")
    chunk: dspy.Image = dspy.InputField(desc="Specific region of interest (chunk) to analyze for defects")

    defects: List[Defect] = dspy.OutputField(
        desc="List of all detected defects in this chunk. Empty list if no defects found."
    )


class LayoutDefectReport(dspy.Signature):
    """
    Aggregate and summarize all defects found across chunks in a layout.
    Produce a clean, complete defect report.
    """
    drawing_summary: str = dspy.InputField(desc="High-level summary of the entire CAD drawing")
    layout_description: str = dspy.InputField(desc="Description of this specific layout")
    all_defects_json: str = dspy.InputField(
        desc="JSON string of all Defect objects found across all chunks in this layout"
    )

    report: CADDefectReport = dspy.OutputField(desc="Final structured defect analysis report")

In [14]:
from collections import defaultdict
import json

# Global dict to collect final reports per layout
all_layout_reports = {}

COVERAGE_THRESHOLD = 0.85

# Use ChainOfThought for better reasoning on both detection and aggregation
chunk_analyzer = dspy.ChainOfThought(DesignDefectAnalysis)
aggregator = dspy.ChainOfThought(LayoutDefectReport)  # Define once, reuse

for l in df['layout_id'].unique():
    layout_img, _ = get_segmentation_crops(
        df, IMAGE_FILE, target_layouts=[l]
    )

    layout_desc = LAYOUT_OUTPUTS[l][0].response
    layout_area = layout_img.width * layout_img.height

    # Collect all defects for this layout
    layout_defects = []
    defect_id_counter = 1

    for c, data in clusters.items():
        # Handle potential missing layout in cluster data
        if l not in data:
            continue
        chunk_ids = data[l]['chunk_ids']
        cluster_id = c
        multi_chunk = len(chunk_ids) > 1

        ext_roi, _ = get_segmentation_crops(
            df, IMAGE_FILE,
            target_layouts=[l],
            target_chunks=chunk_ids
        )

        for ch_id in chunk_ids:
            roi, _ = get_segmentation_crops(
                df, IMAGE_FILE,
                target_layouts=[l],
                target_chunks=[ch_id]
            )

            # Skip chunks that cover too much of the layout (redundant with full layout analysis)
            if multi_chunk:
                ratio = (roi.width * roi.height) / layout_area
                if ratio >= COVERAGE_THRESHOLD:
                    continue

            try:
                prediction = chunk_analyzer(
                    description=output.response,                    # Full drawing context
                    layout_description=layout_desc,
                    layout_image=dspy.Image.from_PIL(layout_img),
                    cluster=dspy.Image.from_PIL(ext_roi),
                    chunk=dspy.Image.from_PIL(roi)
                )

                for defect in prediction.defects:
                    # Use model_copy() instead of deprecated .copy()
                    enriched_defect = defect.model_copy(update={
                        "id": defect_id_counter,
                        "layout_id": str(l),
                        "chunk_id": str(ch_id),
                        "cluster_id": str(cluster_id) if multi_chunk else None
                    })
                    layout_defects.append(enriched_defect)
                    defect_id_counter += 1

                print(f"Layout {l} | Cluster {cluster_id} | Chunk {ch_id}: Found {len(prediction.defects)} defect(s)")

            except Exception as e:
                print(f"Error analyzing Layout {l} | Chunk {ch_id}: {e}")

    # Generate final structured report
    if layout_defects:
        all_defects_json = json.dumps([d.model_dump() for d in layout_defects])

        try:
            final_pred = aggregator(
                drawing_summary=output.response,
                layout_description=layout_desc,
                all_defects_json=all_defects_json
            )
            report = final_pred.report
        except Exception as e:
            print(f"Error during report aggregation for Layout {l}: {e}")
            # Fallback: create manual report
            report = CADDefectReport(
                drawing_summary=output.response,
                defects=layout_defects,
                overall_recommendations="Defects detected but aggregation failed. Review individual findings.",
                sources=["DSPy Chunk Analysis (fallback mode)"]
            )
    else:
        # No defects found
        report = CADDefectReport(
            drawing_summary=f"Layout {l}: {layout_desc[:200]}...",  # Truncated for brevity
            defects=[],
            overall_recommendations="No design defects detected in this layout after detailed chunk analysis.",
            sources=["DSPy Vision Analysis", "Segmented ROI Inspection"]
        )

    all_layout_reports[l] = report
    print(f"\n=== Final Report for Layout {l} ===")
    print(report.model_dump_json(indent=2))

In [6]:
# instructions = "Find all foundation titles relevant to verifying (or refuting) the claim."
# signature = dspy.Signature("claim -> titles: list[str]", instructions)
# react = dspy.ReAct(signature, tools=[retrieve_pages], max_iters=20)